# Pre-Annotate Prescription Sentences (v2 — Dictionary + scispaCy)

Improves on the first pre-annotation pass. scispaCy alone missed several
common US brand-name drugs (e.g. Prevacid, Digitek, Flomax) because it's
trained on general biomedical literature, not prescription text.

This version adds the **`drug_named_entity_recognition`** library (built on
DrugBank's open CC0 data — huge brand + generic name coverage) as the
primary DRUG tagger, with scispaCy as a supplementary pass for anything the
dictionary misses, plus DIAGNOSIS tagging (scispaCy `DISEASE` label).

Regex still handles DOSAGE / FREQUENCY / DURATION (same as before — these
were already close to 100% accurate).

**Output:** `preannotated_tasks_v2.json` — import this into Label Studio
instead of the v1 file for much higher DRUG recall, meaning far fewer manual
additions per sentence.


## 1. Install dependencies

In [1]:
%pip install drug-named-entity-recognition pandas -q
%pip install scispacy -q
%pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bc5cdr_md-0.5.4.tar.gz -q

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import re
import json
import string
import pandas as pd
import spacy
from drug_named_entity_recognition import find_drugs

INPUT_CSV = "../data/processed/annotation_batch_top700.csv"
OUTPUT_JSON = "../data/annotated/preannotated_tasks_v2.json"

nlp = spacy.load("en_ner_bc5cdr_md")
print("scispaCy model loaded.")

C:\Users\rohit\AppData\Local\Programs\Python\Python312\Lib\site-packages\spacy\language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


scispaCy model loaded.


## 2. Regex patterns for DOSAGE / FREQUENCY / DURATION (unchanged)

In [3]:
DOSAGE_RE = re.compile(
    r"\b\d+(\.\d+)?\s?(mg|mcg|ml|g|units?|tablets?|tabs?|sprays?|drops?|puffs?|capsules?|mEq)\b",
    re.IGNORECASE,
)

FREQUENCY_RE = re.compile(
    r"\b(once|twice|three times|four times)\s+(a\s+)?(day|daily)\b"
    r"|\bevery\s+\d+\s+hours?\b"
    r"|\b(b\.i\.d\.?|t\.i\.d\.?|q\.i\.d\.?|q\.d\.?|p\.r\.n\.?|bid|tid|qid|qd|prn)\b"
    r"|\b\d-\d-\d\b"
    r"|\bdaily\b",
    re.IGNORECASE,
)

DURATION_RE = re.compile(
    r"\bfor\s+\d+\s+(day|days|week|weeks|month|months)\b"
    r"|\bfor\s+(a|one|two|three|four|five|six)\s+(day|days|week|weeks|month|months)\b"
    r"|\bx\d+d\b",
    re.IGNORECASE,
)

REGEX_LABEL_MAP = [
    (DOSAGE_RE, "DOSAGE"),
    (DURATION_RE, "DURATION"),
    (FREQUENCY_RE, "FREQUENCY"),
]

def regex_spans(sentence: str):
    spans = []
    claimed = [False] * len(sentence)
    for pattern, label in REGEX_LABEL_MAP:
        for m in pattern.finditer(sentence):
            s, e = m.start(), m.end()
            if any(claimed[s:e]):
                continue
            spans.append((s, e, label))
            for i in range(s, e):
                claimed[i] = True
    return spans

## 3. Dictionary-based DRUG tagger (primary)

Tokenizes the sentence while tracking character offsets, strips surrounding
punctuation per token (so "Prevacid," doesn't drag a trailing comma into the
tag), then looks up each token sequence against the DrugBank-derived
dictionary.

In [4]:
def tokenize_with_offsets(sentence):
    return [(m.start(), m.end(), m.group()) for m in re.finditer(r"\S+", sentence)]

def strip_punct_span(start, end, text):
    lstripped = text.lstrip(string.punctuation)
    rstripped = text.rstrip(string.punctuation)
    new_start = start + (len(text) - len(lstripped))
    new_end = end - (len(text) - len(rstripped))
    return new_start, new_end

def dictionary_drug_spans(sentence, already_claimed):
    tokens = tokenize_with_offsets(sentence)
    clean_words = [t[2].strip(string.punctuation) for t in tokens]
    try:
        results = find_drugs(clean_words, is_ignore_case=True)
    except Exception:
        return []

    spans = []
    for match in results:
        _, start_idx, end_idx = match
        last_idx = end_idx - 1  # end_idx is exclusive
        s_char, _ = strip_punct_span(*tokens[start_idx][:2], tokens[start_idx][2])
        _, e_char = strip_punct_span(*tokens[last_idx][:2], tokens[last_idx][2])
        if any(already_claimed[s_char:e_char]):
            continue
        spans.append((s_char, e_char, "DRUG"))
    return spans

## 4. scispaCy spans (fallback DRUG + DIAGNOSIS)

`CHEMICAL` → `DRUG` (only if the dictionary didn't already catch it),
`DISEASE` → `DIAGNOSIS`.

In [5]:
SCISPACY_LABEL_MAP = {
    "CHEMICAL": "DRUG",
    "DISEASE": "DIAGNOSIS",
}

def model_spans(sentence, already_claimed):
    doc = nlp(sentence)
    spans = []
    for ent in doc.ents:
        mapped_label = SCISPACY_LABEL_MAP.get(ent.label_)
        if not mapped_label:
            continue
        s, e = ent.start_char, ent.end_char
        if any(already_claimed[s:e]):
            continue
        spans.append((s, e, mapped_label))
        for i in range(s, e):
            already_claimed[i] = True
    return spans

## 5. Full pipeline: sanity check on the known-tricky example sentence

In [6]:
example = ("Triglide 50 mg daily, Prevacid 30 mg daily, Plavix 75 mg daily, "
           "potassium 10 mEq daily, Lasix 60 mg daily, folate 1 mg b.i.d., "
           "Niaspan 500 mg daily, atenolol 50 mg daily, enalapril 10 mg b.i.d., "
           "glyburide 10 mg b.i.d., Xanax 0.25 mg b.i.d., aspirin 325 mg daily, "
           "Tylenol p.r.n., Zantac 150 mg b.i.d., Crestor 5 mg daily, TriCor 145 mg daily, "
           "Digitek 0.125 mg daily, Celexa 20 mg daily, and Flomax 0.4 mg daily.")

claimed = [False] * len(example)

r_spans = regex_spans(example)
for s, e, _ in r_spans:
    for i in range(s, e):
        claimed[i] = True

d_spans = dictionary_drug_spans(example, claimed)
for s, e, _ in d_spans:
    for i in range(s, e):
        claimed[i] = True

m_spans = model_spans(example, claimed)

all_spans = sorted(r_spans + d_spans + m_spans, key=lambda x: x[0])
drug_spans = [s for s in all_spans if s[2] == "DRUG"]

print(f"Total spans: {len(all_spans)}  |  DRUG spans found: {len(drug_spans)} (expect 17-19)")
for s, e, label in drug_spans:
    print(f"  {label} -> {example[s:e]!r}")

Total spans: 55  |  DRUG spans found: 18 (expect 17-19)
  DRUG -> 'Prevacid'
  DRUG -> 'Plavix'
  DRUG -> 'potassium'
  DRUG -> 'Lasix'
  DRUG -> 'folate'
  DRUG -> 'Niaspan'
  DRUG -> 'atenolol'
  DRUG -> 'enalapril'
  DRUG -> 'glyburide'
  DRUG -> 'Xanax'
  DRUG -> 'aspirin'
  DRUG -> 'Tylenol'
  DRUG -> 'Zantac'
  DRUG -> 'Crestor'
  DRUG -> 'TriCor'
  DRUG -> 'Digitek'
  DRUG -> 'Celexa'
  DRUG -> 'Flomax'


## 6. Run on all 700 sentences

In [7]:
df = pd.read_csv(INPUT_CSV)
print(f"Loaded {len(df)} sentences.")

tasks = []
drug_counts = []

for _, row in df.iterrows():
    sentence = str(row["sentence"])
    claimed = [False] * len(sentence)

    r_spans = regex_spans(sentence)
    for s, e, _ in r_spans:
        for i in range(s, e):
            claimed[i] = True

    d_spans = dictionary_drug_spans(sentence, claimed)
    for s, e, _ in d_spans:
        for i in range(s, e):
            claimed[i] = True

    m_spans = model_spans(sentence, claimed)

    all_spans = r_spans + d_spans + m_spans
    drug_counts.append(sum(1 for s in all_spans if s[2] == "DRUG"))

    results = []
    for i, (s, e, label) in enumerate(all_spans):
        results.append({
            "id": f"span_{i}",
            "from_name": "label",
            "to_name": "text",
            "type": "labels",
            "value": {"start": s, "end": e, "text": sentence[s:e], "labels": [label]},
        })

    tasks.append({
        "data": {
            "sentence": sentence,
            "file": row.get("file", ""),
            "specialty": row.get("specialty", ""),
            "section": row.get("section", ""),
        },
        "predictions": [{"model_version": "regex+drugdict+scispacy_v2", "result": results}],
    })

print(f"Built {len(tasks)} tasks.")
print(f"Avg DRUG spans per sentence: {sum(drug_counts)/len(drug_counts):.2f}")

Loaded 700 sentences.
Built 700 tasks.
Avg DRUG spans per sentence: 2.43


In [8]:
import os
os.makedirs(os.path.dirname(OUTPUT_JSON), exist_ok=True)

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(tasks, f, ensure_ascii=False, indent=2)

print(f"Saved -> {OUTPUT_JSON}")

Saved -> ../data/annotated/preannotated_tasks_v2.json


## 7. Re-import into Label Studio

1. If you already imported the v1 JSON, you may want to start a **fresh
   project** (or clear existing tasks) to avoid duplicates, then import
   `preannotated_tasks_v2.json`.
2. Same labeling config as before (DRUG, DOSAGE, FREQUENCY, DURATION,
   SYMPTOM, DIAGNOSIS), `value="$sentence"`.
3. Click **Label All Tasks** — DRUG coverage should now be noticeably more
   complete, meaning fewer manual additions per sentence.
4. Still manually add: `SYMPTOM` tags (not auto-tagged at all), and
   spot-check `DIAGNOSIS` (scispaCy's DISEASE tagging is noisier than DRUG).
